In [0]:
# Шлях до  файлів
volume_path = "/Volumes/db_workspace_olena/default/my_data/"

# Читання даних
print("Завантаження даних з Volume...")
flights_df = spark.read.csv(volume_path + "flights.csv", header=True, inferSchema=True)
airports_df = spark.read.csv(volume_path + "airports.csv", header=True, inferSchema=True)
airlines_df = spark.read.csv(volume_path + "airlines.csv", header=True, inferSchema=True)


Завантаження даних з Volume...


In [0]:
# Перевірка на дублікати кодів
airport_counts = airports_df.groupBy("IATA_CODE").count().filter("count > 1")

print("Дублікати кодів у таблиці аеропортів:")
if airport_counts.count() == 0:
    print("Дублікатів не знайдено. Коди унікальні.")
else:
    display(airport_counts)

# Перевірка на дублікати назв (чи має один аеропорт різні коди)
name_counts = airports_df.groupBy("AIRPORT").count().filter("count > 1")
print("Аеропорти з однаковими назвами, але різними кодами:")
if airport_counts.count() == 0:
    print("Дублікатів не знайдено. Відповідність код-назва унікальна.")
else:
    print("Аеропорти з однаковими назвами, але різними кодами:")
    display(name_counts.join(airports_df, "AIRPORT").orderBy("AIRPORT"))

Дублікати кодів у таблиці аеропортів:
Дублікатів не знайдено. Коди унікальні.
Аеропорти з однаковими назвами, але різними кодами:
Дублікатів не знайдено. Відповідність код-назва унікальна.


In [0]:
from pyspark.sql import functions as F

# Знаходимо всі коди з flights, яких немає в airports
# Рахуємо кількість рейсів для кожного такого коду
orphan_stats = flights_df.join(airports_df, flights_df.DESTINATION_AIRPORT == airports_df.IATA_CODE, "left_anti") \
    .groupBy(F.col("DESTINATION_AIRPORT").alias("code")) \
    .agg(F.count("*").alias("flight_count")) \
    .orderBy(F.col("flight_count").desc())

print(f"Всього 'невизначених' кодів: {orphan_stats.count()}")
print("ТОП числових кодів за кількістю рейсів:")
display(orphan_stats)

# Рахуємо загальну кількість зачеплених рядків
total_orphans = orphan_stats.agg(F.sum("flight_count")).collect()[0][0]
print(f"Загальна кількість рейсів з числовими кодами: {total_orphans}")

Всього 'невизначених' кодів: 307
ТОП числових кодів за кількістю рейсів:


code,flight_count
10397,32594
13930,27608
11298,21033
11292,18125
12892,17739
14771,14170
12266,13227
14107,12932
12889,12702
13487,10623


Загальна кількість рейсів з числовими кодами: 486165


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Читаємо обидва довідники (Цифри та Букви)
lookup_id = spark.read.csv("/Volumes/db_workspace_olena/default/my_data/L_AIRPORT_ID.csv", header=True, inferSchema=True)
lookup_code = spark.read.csv("/Volumes/db_workspace_olena/default/my_data/L_AIRPORT.csv", header=True, inferSchema=True)

# Створюємо єдиний реєстр "Код -> Назва"
# Ми беремо частину після двокрапки як назву аеропорту
def prepare_mapping(df):
    return df.select(
        F.col("Code").cast("string").alias("MAPPING_CODE"),
        F.trim(F.substring_index(F.col("Description"), ":", -1)).alias("MAPPING_NAME")
    )

# Об'єднуємо обидва файли в одну таблицю відповідності
mapping_unified = prepare_mapping(lookup_id).union(prepare_mapping(lookup_code)).distinct()

# Приєднуємо цей реєстр до таблиці польотів
# Ми шукаємо відповідність для DESTINATION_AIRPORT
unified_flights = flights_df.join(
    mapping_unified, 
    flights_df.DESTINATION_AIRPORT == mapping_unified.MAPPING_CODE, 
    "left"
).withColumn(
    "AIRPORT_NAME", 
    F.coalesce(F.col("MAPPING_NAME"), F.col("DESTINATION_AIRPORT"))
)

# --- ЗАВДАННЯ 3.3 а: Частина 1 (Дебаг - статистика по всіх аеропортах) ---
print("ЧАСТИНА 1: Статистика для дебагу (всі аеропорти):")
all_stats = unified_flights.groupBy("MONTH", "AIRPORT_NAME").count() \
    .orderBy("MONTH", F.col("count").desc())

display(all_stats) # ПОВНИЙ список

# Збереження
all_stats.orderBy("MONTH").coalesce(1).write.mode("overwrite").option("header", "true").csv("/Volumes/db_workspace_olena/default/my_data/output/monthly_stats")

ЧАСТИНА 1: Статистика для дебагу (всі аеропорти):


MONTH,AIRPORT_NAME,count
1,Hartsfield-Jackson Atlanta International,29492
1,Chicago O'Hare International,23515
1,Dallas/Fort Worth International,23173
1,Los Angeles International,17332
1,Denver International,17059
1,George Bush Intercontinental/Houston,13378
1,Phoenix Sky Harbor International,13121
1,San Francisco International,12883
1,McCarran International,11614
1,Orlando International,10075


In [0]:
# --- ЗАВДАННЯ 3.3 а: Частина 2 (Фінальний звіт - Топ-1 щомісяця) ---
window_spec = Window.partitionBy("MONTH").orderBy(F.col("count").desc())
monthly_leaders = all_stats.withColumn("rank", F.rank().over(window_spec)) \
    .filter(F.col("rank") == 1) \
    .select("MONTH", "AIRPORT_NAME", F.col("count").alias("VISITS"))

print("ЧАСТИНА 2: Найпопулярніші аеропорти:")
display(monthly_leaders.orderBy("MONTH"))

# Збереження
monthly_leaders.orderBy("MONTH").coalesce(1).write.mode("overwrite").option("header", "true").csv("/Volumes/db_workspace_olena/default/my_data/output/popular_airports_final")

ЧАСТИНА 2: Найпопулярніші аеропорти:


MONTH,AIRPORT_NAME,VISITS
1,Hartsfield-Jackson Atlanta International,29492
2,Hartsfield-Jackson Atlanta International,27366
3,Hartsfield-Jackson Atlanta International,32775
4,Hartsfield-Jackson Atlanta International,31383
5,Hartsfield-Jackson Atlanta International,32425
6,Hartsfield-Jackson Atlanta International,32739
7,Hartsfield-Jackson Atlanta International,33735
8,Hartsfield-Jackson Atlanta International,33725
9,Hartsfield-Jackson Atlanta International,31331
10,Hartsfield-Jackson Atlanta International,32594


In [0]:
#  Рахуємо кількість рейсів для кожного коду авіакомпанії
airline_debug_raw = flights_df.groupBy("AIRLINE").count()

#  Приєднуємо повні назви авіакомпаній з airlines_df
airline_debug_full = airline_debug_raw.join(airlines_df, airline_debug_raw.AIRLINE == airlines_df.IATA_CODE) \
    .select(
        airlines_df["AIRLINE"].alias("AIRLINE_FULL_NAME"),
        F.col("count").alias("TOTAL_FLIGHTS")
    ).orderBy(F.col("TOTAL_FLIGHTS").desc())

print("ДЕБАГ: Загальна кількість рейсів на кожну авіакомпанію:")
display(airline_debug_full)

ДЕБАГ: Загальна кількість рейсів на кожну авіакомпанію:


AIRLINE_FULL_NAME,TOTAL_FLIGHTS
Southwest Airlines Co.,1261855
Delta Air Lines Inc.,875881
American Airlines Inc.,725984
Skywest Airlines Inc.,588353
Atlantic Southeast Airlines,571977
United Air Lines Inc.,515723
American Eagle Airlines Inc.,294632
JetBlue Airways,267048
US Airways Inc.,198715
Alaska Airlines Inc.,172521


In [0]:
# Групуємо за авіакомпанією та аеропортом ВІДПРАВЛЕННЯ (ORIGIN_AIRPORT)
cancellation_base = flights_df.groupBy("AIRLINE", "ORIGIN_AIRPORT").agg(
    F.count("*").alias("processed_flights"),
    F.sum("CANCELLED").alias("cancelled_flights")
).withColumn("cancellation_percentage", F.round((F.col("cancelled_flights") / F.col("processed_flights")) * 100, 2))

# Приєднуємо назви авіаліній та аеропортів (використовуючи ваш реєстр mapping_unified)
final_cancellation_report = cancellation_base \
    .join(airlines_df, cancellation_base.AIRLINE == airlines_df.IATA_CODE, "left") \
    .join(mapping_unified, cancellation_base.ORIGIN_AIRPORT == mapping_unified.MAPPING_CODE, "left") \
    .select(
        airlines_df["AIRLINE"].alias("AIRLINE_NAME"), # Беремо повну назву компанії
        F.coalesce(F.col("MAPPING_NAME"), F.col("ORIGIN_AIRPORT")).alias("ORIGIN_AIRPORT_NAME"),
        F.col("cancellation_percentage").alias("PERCENTAGE"),
        F.col("cancelled_flights").alias("CANCELLED_COUNT"),
        F.col("processed_flights").alias("PROCESSED_COUNT")
    )

print("ФІНАЛЬНИЙ ЗВІТ ПО СКАСУВАННЯХ:")
display(final_cancellation_report.orderBy(F.col("PERCENTAGE").desc()).limit(20))

# Збереження у форматі CSV
output_path_cancel = "/Volumes/db_workspace_olena/default/my_data/output/cancellation_rates_final"
final_cancellation_report.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_path_cancel)

print(f"Звіт збережено: {output_path_cancel}")

ФІНАЛЬНИЙ ЗВІТ ПО СКАСУВАННЯХ:


AIRLINE_NAME,ORIGIN_AIRPORT_NAME,PERCENTAGE,CANCELLED_COUNT,PROCESSED_COUNT
American Eagle Airlines Inc.,Gunnison-Crested Butte Regional,27.27,3,11
Atlantic Southeast Airlines,Ithaca Tompkins Regional,16.67,2,12
Frontier Airlines Inc.,Spokane International,16.67,1,6
American Eagle Airlines Inc.,Ronald Reagan Washington National,14.61,110,753
American Eagle Airlines Inc.,Aspen Pitkin County Sardy Field,14.42,77,534
Skywest Airlines Inc.,Juneau International,13.33,8,60
American Eagle Airlines Inc.,Norfolk International,12.53,91,726
American Eagle Airlines Inc.,Salt Lake City International,12.5,9,72
American Eagle Airlines Inc.,Monroe Regional,12.24,6,49
American Eagle Airlines Inc.,Charlottesville Albemarle,11.36,30,264


Звіт збережено: /Volumes/db_workspace_olena/default/my_data/output/cancellation_rates_final
